# 02 — Control de calidad, clustering y tipos celulares

**Taller de célula única CIAD**

Adaptado del *Guided Clustering Tutorial* de Seurat:
<https://satijalab.org/seurat/articles/pbmc3k_tutorial>

Mantén esa página abierta — todo aquí corresponde a ella, así que puedes volver al
original después del taller.

**Los datos.** 2,700 células mononucleares de sangre periférica (PBMCs) de un
donante sano, secuenciadas por 10x Genomics.

**Qué hacemos.** Partir de una matriz de counts y células sin ninguna etiqueta, y terminar con un objeto con los tipos celulares completamente anotados.

1. control de calidad — quitar las células de baja calidad
2. normalización y selección de genes
3. PCA, y después clustering
4. UMAP, para ver el resultado
5. genes marcadores, y nombrar los clusters

**Cómo ejecutar.** *Shift + Enter* corre una celda. Ejecútalas en orden, de arriba
hacia abajo.

Si se desconecta el runtime, vuelve a correr la celda de preparación y luego la
celda de checkpoint de la sección en la que ibas — no tendrás que empezar de
nuevo.

## Preparación

Esto instala Seurat y todo lo demás en la máquina temporal que te dio Colab.
Alrededor de un minuto la primera vez, segundos si la vuelves a correr.

In [ ]:
# This cell installs the packages we need. It is a shortcut for the workshop,
# so that nobody spends the class waiting for an install.
source("https://raw.githubusercontent.com/MartinLoza/CIAD_workshop_sc/main/setup/setup.R")

In [ ]:
# Change R language to English, in case your computer uses another language.
Sys.setenv(LANGUAGE = "en")

# This is the normal way to load a package in R. You will write lines like
# these at the top of every script you make.
library(Seurat)
library(ggplot2)
library(dplyr)
library(patchwork)

# Size of every figure in this notebook, in inches. Change these two numbers
# if a plot looks too small or too large.
options(repr.plot.width = 10, repr.plot.height = 7)

## 1. Los datos

Usamos los mismos datos que el notebook 01: pbmc3k, de 10x Genomics. Llegan en
tres archivos — `matrix.mtx`, `barcodes.tsv` y `genes.tsv`. El notebook 01 mira
dentro de ellos.

Nota que esto lo descarga la máquina de Colab, no tu laptop.

In [ ]:
url <- "https://cf.10xgenomics.com/samples/cell/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"

download.file(url, "pbmc3k.tar.gz", quiet = TRUE)
untar("pbmc3k.tar.gz")

list.files("filtered_gene_bc_matrices/hg19")

`Read10X()` lee esos tres archivos en una sola matriz dispersa (una matriz que
guarda solo los valores que no son cero): genes en las filas, células en las columnas.

La mayoría de las entradas son cero. Un gen simplemente no se detecta en una célula
dada. Guardar solo los valores que no son cero es lo que hace que esta matriz quepa en memoria.

In [ ]:
pbmc.data <- Read10X(data.dir = "filtered_gene_bc_matrices/hg19")

cat("genes:", nrow(pbmc.data), "\n")
cat("cells:", ncol(pbmc.data), "\n\n")

# the names come straight from genes.tsv and barcodes.tsv
cat("first genes:", head(rownames(pbmc.data), 4), "\n")
cat("first cells:", head(colnames(pbmc.data), 2), "\n\n")

# a corner of the matrix: "." is a zero that is not stored
pbmc.data[c("CD3D", "TCL1A", "MS4A1"), 1:20]

### El objeto Seurat

`CreateSeuratObject()` envuelve la matriz junto con todo lo que estamos por
calcular — métricas de calidad, clusters, coordenadas UMAP — en un solo objeto.

Aquí se aplican dos primeros filtros. No son estrictos, así que no perdemos
muchas células ni muchos genes:

- `min.cells = 3` — descarta genes presentes en menos de 3 células. No aportan
  información y solo cuestan memoria.
- `min.features = 200` — descarta gotas (células) con menos de 200 genes detectados.
  Estas células probablemente son células muriendo cuyo RNA ya está degradado (entre muchas otras razones). Sus counts pueden ser ruidosos, así que es mejor quitarlas para que no creen artefactos en nuestros resultados.

In [ ]:
pbmc <- CreateSeuratObject(
  counts       = pbmc.data,
  project      = "pbmc3k",
  min.cells    = 3,
  min.features = 200
)

pbmc

## 2. Control de calidad

No podemos ver cada célula en el microscopio, así que las juzgamos por sus
counts (transcritos). Tres números nos ayudan a evaluar una célula:

| métrica | qué es | qué sugiere un valor extremo |
|---|---|---|
| `nFeature_RNA` | número de genes detectados en la célula | muy bajo: gota vacía o célula muriendo. muy alto: dos células en una gota |
| `nCount_RNA` | total de moléculas (transcritos) en la célula | lo mismo que arriba |
| `percent.mt` | % de counts de genes mitocondriales | alto: la membrana se rompió, el RNA citoplásmico se fugó, el mitocondrial se quedó |

Las dos primeras las calcula `CreateSeuratObject()` por ti. La tercera la agregamos
nosotros: los símbolos de los genes mitocondriales humanos empiezan todos con `MT-`.

In [ ]:
pbmc[["percent.mt"]] <- PercentageFeatureSet(pbmc, pattern = "^MT-")

head(pbmc@meta.data, 5)

### Ver las distribuciones

Una gráfica de violín por métrica. Buscamos dos cosas:

- dónde está la mayoría de las células
- las colas, que contienen las células que quizá queramos quitar

In [ ]:
VlnPlot(pbmc,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        ncol     = 3)

Las métricas son más fáciles de juzgar de dos en dos.

- panel izquierdo: ¿las células con muchos counts también tienen contenido mitocondrial alto?
- panel derecho: ¿las células con muchos counts tienen más genes?

El panel derecho debería mostrar una relación cercana. Las células lejos de ella son
sospechosas.

In [ ]:
p1 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "percent.mt")
p2 <- FeatureScatter(pbmc, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")

p1 + p2

### Filtrado

Los umbrales del tutorial: más de 200 y menos de 2,500 genes, menos de
5% de contenido mitocondrial.

Estos números **no son universales**. Salen de mirar las gráficas de arriba,
para este tejido y este protocolo. En otro conjunto de datos volverías a mirar.

In [ ]:
cells_before <- ncol(pbmc)

pbmc <- subset(pbmc,
               subset = nFeature_RNA > 200 &
                        nFeature_RNA < 2500 &
                        percent.mt   < 5)

cat("before:", cells_before, "cells\n")
cat("after :", ncol(pbmc), "cells\n")
cat("removed:", cells_before - ncol(pbmc),
    sprintf("(%.1f%%)\n", 100 * (cells_before - ncol(pbmc)) / cells_before))

### ✏️ Ejercicio 1

¿Qué tan sensible es esa decisión?

Vuelve a correr el filtro sobre una copia del objeto con un corte mitocondrial
**más estricto** de 2.5%, y reporta cuántas células más pierdes. No sobrescribas `pbmc`.

Llena los espacios en blanco:

In [ ]:
# unf <- ____
# strict <- subset(unf,
#                  subset = nFeature_RNA > 200 &
#                           nFeature_RNA < 2500 &
#                           percent.mt   < ______)
# ncol(strict)

# We no longer have the unfiltered object — pbmc was overwritten above.
# How can you get it back?

## 3. Normalización

### Por qué la necesitamos

Dos células del mismo tipo pueden dar counts muy distintos. Una célula puede
capturarse mejor, o secuenciarse más profundo, que otra. Esto es técnico. No dice
nada sobre la biología de la célula.

Primero vemos qué tan grande es esa diferencia aquí. `nCount_RNA` es el total
de counts en una célula, así que es una medida de la profundidad de secuenciación.

In [ ]:
depth <- pbmc$nCount_RNA

cat("total counts per cell\n")
cat("  smallest cell:", min(depth), "\n")
cat("  median cell  :", median(depth), "\n")
cat("  largest cell :", max(depth), "\n")
cat("  the largest cell has", round(max(depth) / min(depth), 1),
    "times more counts than the smallest one\n")

La célula más grande tiene muchas veces más counts que la más pequeña. Si comparamos
células ahora, comparamos sobre todo profundidad de secuenciación, no biología.

`NormalizeData()` quita esto. Para cada célula hace tres cosas:

1. divide cada count de la célula entre el total de counts de esa célula
2. multiplica por 10,000, un número fijo, para que los valores sean más fáciles de leer
3. toma `log(x + 1)`

El paso 1 quita la profundidad. El paso 3 acerca los valores muy grandes al resto,
para que unos pocos valores muy altos no dominen.

El resultado va a un layer nuevo llamado `data`. Los counts crudos se quedan en el
layer `counts` y no se modifican.

In [ ]:
pbmc <- NormalizeData(pbmc,
                      normalization.method = "LogNormalize",
                      scale.factor         = 10000)

# raw counts and normalised values now live side by side
Layers(pbmc[["RNA"]])

### Un gen, antes y después

Los números se entienden mejor cuando los puedes ver. Sigamos un gen a través de
la normalización.

`ACTB` (beta-actina) es un buen gen para esto. Se expresa en la mayoría de las células, así que
podemos esperar que cada célula tenga un valor para este gen.

In [ ]:
gene <- "ACTB"   # beta-actin, expressed in most cells

gene_df <- data.frame(
  depth = pbmc$nCount_RNA,
  raw   = as.numeric(LayerData(pbmc, assay = "RNA", layer = "counts")[gene, ]),
  norm  = as.numeric(LayerData(pbmc, assay = "RNA", layer = "data")[gene, ])
)

p1 <- ggplot(gene_df, aes(x = raw)) +
  geom_histogram(bins = 50) +
  labs(title = paste(gene, "before normalisation"),
       x = "raw counts in the cell", y = "number of cells") 

p2 <- ggplot(gene_df, aes(x = norm)) +
  geom_histogram(bins = 50) +
  labs(title = paste(gene, "after normalisation"),
       x = "normalised value", y = "number of cells") 

(p1 + p2) & theme_minimal() & theme(plot.title = element_text(hjust = 0.5))

Como habrás notado, además de normalizar por profundidad de secuenciación, la forma de la distribución cambia. Los counts crudos tienen una cola larga a la
derecha. Los valores normalizados son más simétricos. Esto puede ser útil para análisis posteriores que asumen una distribución normal en los datos.

In [ ]:
cat("correlation with sequencing depth\n")
cat("  before:", round(cor(gene_df$depth, gene_df$raw),  2), "\n")
cat("  after :", round(cor(gene_df$depth, gene_df$norm), 2), "\n")

p3 <- ggplot(gene_df, aes(x = depth, y = raw)) +
  geom_point(size = 0.3, alpha = 0.3) +
  labs(title = "before", x = "total counts in the cell",
       y = paste(gene, "raw counts"))

p4 <- ggplot(gene_df, aes(x = depth, y = norm)) +
  geom_point(size = 0.3, alpha = 0.3) +
  labs(title = "after", x = "total counts in the cell",
       y = paste(gene, "normalised value"))

(p3 + p4) & theme_minimal() & theme(plot.title = element_text(hjust = 0.5))

En el panel izquierdo los puntos suben de izquierda a derecha: las células con más counts
totales tienen más counts de `ACTB`. En el panel derecho esa tendencia es mucho más débil,
y el número de correlación es mucho más pequeño.

Esto es lo que queríamos. Lo que queda se parece más a cuánto `ACTB` expresa
realmente la célula, y no a qué tan profundo se secuenció.

**Pruébalo:** cambia `gene` a `"MS4A1"` y vuelve a correr las dos celdas. `MS4A1` es un
marcador de células B, así que la mayoría de las células no lo expresan en absoluto. Las gráficas se ven muy
distintas. ¿Por qué?

## 4. Genes variables

Muchos genes se expresan a un nivel parecido en todas las células (por ejemplo los genes housekeeping). Agregan
tiempo de cómputo, y no ayudan a separar bien los tipos celulares.

`FindVariableFeatures()` se queda con los 2,000 genes con la varianza (variabilidad) más alta para
su valor promedio. Este es el método `vst`. Escoger genes solo por su varianza se enfocaría únicamente en genes con expresión alta.

In [ ]:
pbmc <- FindVariableFeatures(pbmc, selection.method = "vst", nfeatures = 2000)

top10 <- head(VariableFeatures(pbmc), 10)
top10

In [ ]:
p <- VariableFeaturePlot(pbmc)
LabelPoints(plot = p, points = top10, repel = TRUE)

Mira los resultados: `PPBP` (plaquetas), `LYZ` (monocitos), `GNLY` y
`NKG7` (células NK), `S100A8` (neutrófilos y monocitos). El método no sabe nada
de inmunología. Aun así, los genes más variables son marcadores de los tipos celulares
de la muestra. Esa es la importancia de este paso.

## 5. Escalado

El PCA lo maneja la varianza. Sin escalar, un gen con expresión alta dominaría
solo porque sus valores son grandes. `ScaleData()` centra cada gen en
cero y lo escala a varianza uno, para que todos los genes se comparen en igualdad de condiciones.

Aquí escalamos todos los genes, lo que toma unos segundos. Por defecto solo se escalan
los genes variables — suficiente para el PCA, pero los heatmaps de más adelante se ven mejor con
todo escalado.

In [ ]:
pbmc <- ScaleData(pbmc, features = rownames(pbmc))

## 6. PCA

2,000 genes variables siguen siendo demasiadas dimensiones. El PCA los comprime en unos
pocos componentes que capturan la estructura, y son esos componentes —
no los genes — los que usan el clustering y el UMAP.

In [ ]:
pbmc <- RunPCA(pbmc, features = VariableFeatures(pbmc), verbose = FALSE)

print(pbmc[["pca"]], dims = 1:5, nfeatures = 5)

Cada componente es una combinación ponderada de genes. Arriba están impresos los
cinco genes con mayor peso en cada uno de los primeros cinco componentes. Otra vez
parecen firmas de tipos celulares.

Un heatmap lo hace concreto. Las células están ordenadas por su valor en el componente,
y los genes por su loading. Una estructura de bloques limpia significa que el componente separa
algo real.

In [ ]:
DimHeatmap(pbmc, dims = 1:6, cells = 500, balanced = TRUE)

### ¿Cuántos componentes?

Si te quedas con muy pocos pierdes estructura real. Con demasiados haces clustering sobre ruido.

La gráfica de codo (elbow plot) muestra cuánta varianza explica cada componente. Donde la curva
se aplana, los componentes restantes son sobre todo ruido.

In [ ]:
ElbowPlot(pbmc, ndims = 30)

### ✏️ Ejercicio 2

Mira la gráfica de codo y decide dónde se aplana.

El tutorial usa 10. ¿Se puede defender eso con la gráfica? ¿Cambiaría mucho con 15?

Pon abajo en `n_dims` el número que defenderías, y lo usaremos para el
resto del notebook.

In [ ]:
n_dims <- 10   # replace with your choice, e.g. 10
#cat("using", n_dims, "principal components\n")

## 7. Clustering

Dos pasos.

`FindNeighbors()` construye un grafo: cada célula se conecta con las células más cercanas a
ella en el espacio del PCA.

`FindClusters()` luego encuentra grupos de células que están más conectadas entre sí
que con el resto — comunidades en ese grafo.

`resolution` controla qué tan fino es el clustering. Un valor más alto da más
clusters, y cada uno es más pequeño. No hay un valor correcto. 0.4–1.2 es el
rango usual para unos pocos miles de células.

In [ ]:
pbmc <- FindNeighbors(pbmc, dims = 1:n_dims)
pbmc <- FindClusters(pbmc, resolution = 0.5)

table(Idents(pbmc))

## 8. UMAP

UMAP coloca cada célula en un mapa de 2D, tratando de mantener como vecinas en el mapa
a las células que eran vecinas en el espacio del PCA.

Dos advertencias:

- **La distancia entre clusters significa poco.** Dos clusters lejanos en un UMAP
  no son necesariamente más distintos que dos que están cerca.
- **El UMAP no define los clusters.** Los clusters se calcularon en el espacio del PCA,
  en `n_dims` dimensiones. El UMAP solo los dibuja.

In [ ]:
pbmc <- RunUMAP(pbmc, dims = 1:n_dims, verbose = FALSE)

DimPlot(pbmc, reduction = "umap", label = TRUE) + NoLegend()

### 💾 Checkpoint

Si vas atrasado, o si el runtime se cayó, este es un buen punto para guardar.

La celda de abajo escribe el objeto en el disco de la máquina de Colab. Desaparece cuando
el runtime se apaga, pero sobrevive a que vuelvas a correr celdas por accidente.

In [ ]:
saveRDS(pbmc, "pbmc_clustered.rds")

# To come back to this point later:
# pbmc <- readRDS("pbmc_clustered.rds")

cat("saved:", round(file.size("pbmc_clustered.rds") / 1e6, 1), "MB\n")

## 9. Genes marcadores

Tenemos clusters con números. Para nombrarlos necesitamos saber qué expresa cada uno
que los demás no.

`FindAllMarkers()` compara cada cluster contra todas las células restantes, un
cluster a la vez. `only.pos = TRUE` se queda solo con los genes que están *más altos* en el
cluster, que es lo que queremos para nombrar un cluster.

Esta es probablemente la celda más lenta del notebook...

In [ ]:
markers <- FindAllMarkers(pbmc, only.pos = TRUE, verbose = FALSE)

markers %>%
  group_by(cluster) %>%
  slice_max(avg_log2FC, n = 3) %>%
  ungroup() %>%
  as.data.frame()

Dos formas de ver un marcador: dónde se expresa en el mapa, y con qué
fuerza se expresa por cluster.

In [ ]:
FeaturePlot(pbmc,
            features = c("MS4A1", "CD3E", "CD14", "FCGR3A",
                         "GNLY", "LYZ", "FCER1A", "PPBP"),
            ncol     = 4)

In [ ]:
VlnPlot(pbmc, features = c("MS4A1", "CD3E", "CD14", "PPBP"), ncol = 2)

Un heatmap de los mejores marcadores por cluster muestra todo a la vez.
Cada columna es una célula, agrupadas por cluster; cada fila es un gen.

In [ ]:
top_markers <- markers %>%
  group_by(cluster) %>%
  dplyr::filter(avg_log2FC > 1) %>%
  slice_head(n = 10) %>%
  ungroup()

DoHeatmap(pbmc, features = top_markers$gene) + NoLegend()

## 10. Nombrar los clusters

Este es el paso que ningún algoritmo hace por ti. Tú relacionas los genes marcadores con
la biología conocida.

Para PBMCs los marcadores canónicos están bien establecidos:

| marcadores | tipo celular |
|---|---|
| `IL7R`, `CCR7` | T CD4 naive |
| `IL7R`, `S100A4` | T CD4 de memoria |
| `CD14`, `LYZ` | monocitos CD14+ |
| `MS4A1` | B |
| `CD8A` | T CD8 |
| `FCGR3A`, `MS4A7` | monocitos FCGR3A+ |
| `GNLY`, `NKG7` | NK |
| `FCER1A`, `CST3` | células dendríticas |
| `PPBP` | plaquetas |

### ✏️ Ejercicio 3

Las etiquetas de abajo son las del tutorial, en el orden de clusters del tutorial. **Tu
numeración de clusters puede ser distinta** — depende del número de componentes y de la
resolución que escogiste.

Revísalas contra tus propios marcadores antes de aceptarlas. Si tu cluster 3 no es
el cluster de células B, el vector está mal para ti y hay que reordenarlo.

In [ ]:
# The order must match levels(pbmc) — check first:
levels(pbmc)

In [ ]:
new_ids <- c("Naive CD4 T", "CD14+ Mono", "Memory CD4 T", "B",
             "CD8 T", "FCGR3A+ Mono", "NK", "DC", "Platelet")

# only works if you have exactly as many clusters as labels
stopifnot(length(new_ids) == length(levels(pbmc)))

names(new_ids) <- levels(pbmc)
pbmc <- RenameIdents(pbmc, new_ids)

DimPlot(pbmc, reduction = "umap", label = TRUE, pt.size = 0.5) + NoLegend()

Guarda las etiquetas en un lugar permanente. `Idents()` es fácil de sobrescribir por
accidente; una columna de metadata no.

In [ ]:
pbmc$cell_type <- Idents(pbmc)

table(pbmc$cell_type)

In [ ]:
saveRDS(pbmc, "pbmc_annotated.rds")

cat("saved:", round(file.size("pbmc_annotated.rds") / 1e6, 1), "MB\n")

## Qué hicimos

De una matriz de counts sin etiquetas, a tipos celulares inmunes nombrados, en unos diez
pasos.

Vale la pena llevarse esto:

- **Los umbrales de control de calidad son un juicio, no valores por defecto.** Miramos las distribuciones y
  después escogimos. Otro tejido, otros números.
- **El clustering ocurre en el espacio del PCA, no en el UMAP.** La imagen es una
  proyección del resultado, no el resultado.
- **La resolución y el número de dimensiones son decisiones.** Cambian cuántos clusters
  obtienes. Si un resultado aparece con una sola resolución, no confíes en él.
- **Nombrar los clusters es la biología.** Los pasos anteriores preparan los datos.

Siguiente notebook: qué pasa cuando los datos vienen de más de una muestra, y
el batch effect es más grande que la biología.

---

### Respuestas

<details>
<summary>Clic para desplegar</summary>

**Ejercicio 1**

`pbmc` se sobrescribió en el paso de filtrado, así que el objeto sin filtrar ya no está.
La forma más barata de recuperarlo es reconstruirlo desde `pbmc.data`, que sigue en memoria:

```r
unf <- CreateSeuratObject(pbmc.data, min.cells = 3, min.features = 200)
unf[["percent.mt"]] <- PercentageFeatureSet(unf, pattern = "^MT-")
strict <- subset(unf, subset = nFeature_RNA > 200 &
                               nFeature_RNA < 2500 &
                               percent.mt   < 2.5)
ncol(strict)
```

La lección: si asignas el objeto filtrado a su propio nombre, pierdes el
objeto que necesitas para revisar el filtro. Usa un nombre nuevo.

**Ejercicio 2**

Cualquier valor entre 10 y 15 se puede defender. La curva está claramente plana pasando ~15, y
la diferencia entre 10 y 15 es pequeña. Ese es el punto. Si tus clusters
cambian por completo entre 10 y 15 componentes, no eran estables.

**Ejercicio 3**

Compara los mejores marcadores de cada cluster con la tabla de arriba. Con 10 componentes a
resolución 0.5 normalmente obtienes 9 clusters en el orden del tutorial, pero esto no
está garantizado. Si `stopifnot()` falla, tienes un número distinto de clusters —
baja la resolución, o escribe un vector de etiquetas que corresponda a lo que realmente tienes.

</details>